In [1]:
import pandas as pd

df_raw = pd.read_csv('financial_rawdata.csv')

meta_cols = ['tic', 'datadate', 'gvkey']
core_financials = [
    'atq', 'revtq', 'cogsq', 'niq', 'oibdpq', 'xrdq', 'xsgaq',
    'cheq', 'actq', 'lctq', 'dlcq', 'dlttq', 'ceqq', 'cshoq', 'prccq'
]

numeric_cols = [col for col in df_raw.columns if col not in meta_cols]

threshold = 0.3 * len(df_raw)
cols_to_keep_for_pca = []

for col in numeric_cols:
    if col in core_financials or df_raw[col].isnull().sum() < threshold:
        cols_to_keep_for_pca.append(col)

final_cols = meta_cols + cols_to_keep_for_pca
df_clean = df_raw[final_cols].copy()

if 'xrdq' in df_clean.columns:
    df_clean['xrdq'] = df_clean['xrdq'].fillna(0)

df_clean['datadate'] = pd.to_datetime(df_clean['datadate'])

# keep the latest one
df_latest = df_clean.sort_values(by=['tic', 'datadate']).drop_duplicates(subset=['tic'], keep='last').copy()

output_name = 'Finance_Features_Latest_Only.csv'
df_latest.to_csv(output_name, index=False)

In [2]:
import re
import numpy as np

INPUT_PATH = "Finance_Features_Latest_Only.csv"
OUT_RAW = "cleaned_quarterly_raw.csv"
OUT_FEAT = "finance_features_ml_ready.csv"

WINSOR_P = (0.01, 0.99)
DROP_MISSING_COL_THRESH = 0.70
FILL_NUMERIC_WITH = "median"


In [3]:
def clean_ticker(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).upper().strip()
    x = re.sub(r"\s+", "", x)
    return x

def safe_div(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    out = a / b
    if np.isinf(out) or pd.isna(out):
        return np.nan
    return float(out)

def winsorize_series(s: pd.Series, p_low=0.01, p_high=0.99) -> pd.Series:
    if s.dropna().empty:
        return s
    lo = s.quantile(p_low)
    hi = s.quantile(p_high)
    return s.clip(lower=lo, upper=hi)

def safe_log(x):
    x = pd.to_numeric(x, errors="coerce")
    if pd.isna(x) or x <= 0:
        return np.nan
    return float(np.log(x))

In [4]:
# 1) Load
df = pd.read_csv(INPUT_PATH)
print("Loaded shape:", df.shape)

# Basic standardization
if "tic" in df.columns:
    df["tic"] = df["tic"].apply(clean_ticker)

# Parse dates
for c in ["datadate"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

Loaded shape: (403, 165)


In [5]:
# 2) Deduplicate
key_cols = [c for c in ["gvkey", "datadate"] if c in df.columns]
if len(key_cols) == 2:
    before = len(df)
    df = df.sort_values(key_cols).drop_duplicates(subset=key_cols, keep="last")
    print("Deduped:", before, "->", len(df))

Deduped: 403 -> 403


In [6]:
# 3) Convert numerics
id_like = set(["tic", "gvkey", "costat", "curcdq", "datafmt", "indfmt", "consol",
               "datacqtr", "datafqtr", "fyr", "datadate"])

for c in df.columns:
    if c not in id_like:
        df[c] = pd.to_numeric(df[c], errors="coerce")

In [18]:
# 4) Drop useless columns (all missing / high missingness)
missing_rate = df.isna().mean().sort_values(ascending=False)

all_missing_cols = missing_rate[missing_rate == 1.0].index.tolist()
if all_missing_cols:
    df = df.drop(columns=all_missing_cols)
    print("Dropped all-missing cols:", len(all_missing_cols))

high_missing_cols = missing_rate[(missing_rate > DROP_MISSING_COL_THRESH) & (missing_rate < 1.0)].index.tolist()

core_keep = {"atq","saleq","revtq","niq","oiadpq","oibdpq","cheq","dlttq","dlcq","ltq","seqq","ceqq",
             "oancfq","capxq","xrdq","prccq","cshoq","actq","lctq","wcapq", "cogsq","intanq","dpq"}
high_missing_cols = [c for c in high_missing_cols if c not in core_keep]

if high_missing_cols:
    df = df.drop(columns=high_missing_cols)
    print("Dropped high-missing cols (>%.0f%%):" % (DROP_MISSING_COL_THRESH*100), len(high_missing_cols))

In [19]:
# 5) Winsorize numeric columns (to reduce outlier influence)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

no_winsor = set([c for c in ["gvkey", "fyr"] if c in df.columns])
wcols = [c for c in num_cols if c not in no_winsor]

p_low, p_high = WINSOR_P
for c in wcols:
    df[c] = winsorize_series(df[c], p_low, p_high)

In [20]:
# 6) Build finance features (core ratios / size / valuation)
def get_col(name, fallback=None):
    if name in df.columns:
        return df[name]
    return fallback

# Choose revenue column
rev = get_col("saleq", get_col("revtq"))

# Debt
dltt = get_col("dlttq")
dlc = get_col("dlcq")
debt_total = (pd.to_numeric(dltt, errors="coerce") if dltt is not None else np.nan) + \
             (pd.to_numeric(dlc, errors="coerce") if dlc is not None else np.nan)

# Equity preference: seqq then ceqq
equity = get_col("seqq", get_col("ceqq"))

# FCF
oancf = get_col("oancfq")
capx = get_col("capxq")
fcf = (pd.to_numeric(oancf, errors="coerce") if oancf is not None else np.nan) - \
      (pd.to_numeric(capx, errors="coerce") if capx is not None else np.nan)

# Market cap from Compustat quarter-end price * shares outstanding
prccq = get_col("prccq")
cshoq = get_col("cshoq")
mktcap = pd.to_numeric(prccq, errors="coerce") * pd.to_numeric(cshoq, errors="coerce")

at = get_col("atq")
ni = get_col("niq")
oi = get_col("oiadpq")
cash = get_col("cheq")
xrd = get_col("xrdq")
act = get_col("actq")
lct = get_col("lctq")

cogs = get_col("cogsq")
intan = get_col("intanq")
capx_raw = get_col("capxq")

features = pd.DataFrame({
    "tic": df["tic"] if "tic" in df.columns else np.nan,
    "gvkey": df["gvkey"] if "gvkey" in df.columns else np.nan,
    "datadate": df["datadate"] if "datadate" in df.columns else pd.NaT,

# raw essentials (useful for audit)
    "assets": at,
    "revenue": rev,
    "net_income": ni,
    "operating_income": oi,
    "cash": cash,
    "debt_total": debt_total,
    "equity": equity,
    "oancf": oancf,
    "capx": capx,
    "fcf": fcf,
    "rd": xrd,
    "mktcap": mktcap,

     "cogs": cogs,
    "intangibles": intan,
    "current_assets": act,
    "current_liabilities": lct,
    "long_term_debt": dltt,
})



In [21]:
# Ratios / engineered
features["roa"] = features.apply(lambda r: safe_div(r["net_income"], r["assets"]), axis=1)
features["roe"] = features.apply(lambda r: safe_div(r["net_income"], r["equity"]), axis=1)
features["op_margin"] = features.apply(lambda r: safe_div(r["operating_income"], r["revenue"]), axis=1)
features["net_margin"] = features.apply(lambda r: safe_div(r["net_income"], r["revenue"]), axis=1)

features["cash_to_assets"] = features.apply(lambda r: safe_div(r["cash"], r["assets"]), axis=1)
features["ocf_to_assets"] = features.apply(lambda r: safe_div(r["oancf"], r["assets"]), axis=1)
features["fcf_to_assets"] = features.apply(lambda r: safe_div(r["fcf"], r["assets"]), axis=1)

features["debt_to_assets"] = features.apply(lambda r: safe_div(r["debt_total"], r["assets"]), axis=1)
features["debt_to_equity"] = features.apply(lambda r: safe_div(r["debt_total"], r["equity"]), axis=1)


In [22]:
# Liquidity ratios (if act/lct exist)
if act is not None and lct is not None:
    features["current_ratio"] = features.apply(lambda r: safe_div(getattr(r, "act", np.nan), getattr(r, "lct", np.nan)), axis=1)
else:
    features["current_ratio"] = np.nan

features["rd_to_sales"] = features.apply(lambda r: safe_div(r["rd"], r["revenue"]), axis=1)
features["capx_to_sales"] = features.apply(lambda r: safe_div(r["capx"], r["revenue"]), axis=1)

features["log_assets"] = features["assets"].apply(safe_log)
features["log_mktcap"] = features["mktcap"].apply(safe_log)

features["mkt_to_book"] = features.apply(lambda r: safe_div(r["mktcap"], r["equity"]), axis=1)
features["p_to_s"] = features.apply(lambda r: safe_div(r["mktcap"], r["revenue"]), axis=1)
features["p_to_e"] = features.apply(lambda r: safe_div(r["mktcap"], r["net_income"]), axis=1)  # NI<=0会不稳定/NaN，正常


In [23]:
# new features
# 1. gross_margin
features["gross_margin"] = features.apply(
    lambda r: safe_div(r["revenue"] - r["cogs"], r["revenue"]), axis=1)
# 2. asset_turnover
features["asset_turnover"] = features.apply(
    lambda r: safe_div(r["revenue"], r["assets"]), axis=1)
# 3. intangibles_to_assets
features["intangibles_to_assets"] = features.apply(
    lambda r: safe_div(r["intangibles"], r["assets"]), axis=1)
# 4. capex_to_sales
features["capex_to_sales"] = features.apply(
    lambda r: safe_div(r["capx"], r["revenue"]), axis=1)
# 5. current_ratio
features["current_ratio"] = features.apply(
    lambda r: safe_div(r["current_assets"], r["current_liabilities"]), axis=1)
# 6. cash_ratio
features["cash_ratio"] = features.apply(
    lambda r: safe_div(r["cash"], r["current_liabilities"]), axis=1)
# 7. long_term_debt_ratio
features["long_term_debt_ratio"] = features.apply(
    lambda r: safe_div(r["long_term_debt"], r["assets"]), axis=1)

In [24]:
# 7) Create missing flags
feat_num_cols = features.select_dtypes(include=[np.number]).columns.tolist()
feat_id_cols = ["gvkey"] if "gvkey" in features.columns else []
feat_num_cols = [c for c in feat_num_cols if c not in feat_id_cols]

# Missing indicators (for explainability)
for c in feat_num_cols:
    features[f"{c}_missing"] = features[c].isna().astype(int)

# Impute
if FILL_NUMERIC_WITH == "median":
    fill_vals = features[feat_num_cols].median(numeric_only=True)
else:
    fill_vals = features[feat_num_cols].mean(numeric_only=True)

features[feat_num_cols] = features[feat_num_cols].fillna(fill_vals)


In [25]:
# delete all columns with NaN
all_nan_cols = features.columns[features.isna().all()].tolist()

print("All-NaN columns:", len(all_nan_cols))
print(all_nan_cols)

features = features.drop(columns=all_nan_cols)

All-NaN columns: 7
['oancf', 'capx', 'fcf', 'ocf_to_assets', 'fcf_to_assets', 'capx_to_sales', 'capex_to_sales']


In [26]:
# delete all columns with *_missing
missing_cols = [c for c in features.columns if c.endswith("_missing")]
features = features.drop(columns=missing_cols)

print("Dropped missing indicator columns:", len(missing_cols))

print("Final shape:", features.shape)

Dropped missing indicator columns: 38
Final shape: (403, 36)


In [29]:
# 8) Save outputs
df.to_csv(OUT_RAW, index=False)
features.to_csv(OUT_FEAT, index=False)

print("\nSaved:")
print(" -", OUT_RAW, "shape:", df.shape)
print(" -", OUT_FEAT, "shape:", features.shape)


Saved:
 - cleaned_quarterly_raw.csv shape: (403, 172)
 - finance_features_ml_ready.csv shape: (403, 36)
